<a href="https://colab.research.google.com/github/pragadeesh1024/menta-health-prediction/blob/main/mental_health_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
from nltk.stem import WordNetLemmatizer
import nltk
import re


In [ ]:
df=pd.read_csv('/content/mental_health_combined_test.csv')

In [ ]:
df

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

In [ ]:
def process(text):
  text=text.lower()
  text=re.sub(r'[^a-z0-9\s]','',text)
  stop_words=set(stopwords.words('english'))
  word_tokens=text.split()
  word_1=[w for w in word_tokens if not w in stop_words]
  l=WordNetLemmatizer()
  lm=[l.lemmatize(w) for w in word_1]
  return ' '.join(lm)
  df['clean']=df['text'].apply(process)



In [ ]:
df['clean'] = df['text'].apply(process)
df['clean']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer()
x=tf.fit_transform(df['clean'])
y=tf.fit_transform(df['status'])

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:

x_train = x_train.toarray()
x_test = x_test.toarray()

In [ ]:
# 'vocab' should represent the number of features (vocabulary size)
# from the TfidfVectorizer, which is the second dimension of x_train (or x_test).
vocab = x_train.shape[1]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,SimpleRNN,LSTM
model=Sequential()
model.add(SimpleRNN(input_shape=(1,vocab),units=128,activation='relu'))
model.add(Dense(4,activation='softmax'))

In [ ]:
# Reshape x_train and x_test to match the SimpleRNN input requirements
# Expected shape: (samples, timesteps, features)
# Current x_train/x_test shape: (samples, features)
x_train = x_train.reshape(x_train.shape[0], 1, x_train.shape[1])
x_test = x_test.reshape(x_test.shape[0], 1, x_test.shape[1])

# Convert y_train and y_test from sparse matrices to dense arrays
# The model expects one-hot encoded labels for categorical_crossentropy
y_train = y_train.toarray()
y_test = y_test.toarray()

model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.fit(x_train,y_train,epochs=10,batch_size=32,validation_data=(x_test,y_test))

### Predict Mental Health Status for New Input

In [ ]:
# Get input text from the user
input_text = input("Enter text for prediction: ")

In [ ]:
# Preprocess the input text using the same 'process' function
cleaned_input_text = process(input_text)
print(f"Cleaned input: {cleaned_input_text}")

In [ ]:

input_features = tf.transform([cleaned_input_text])


input_features_dense = input_features.toarray()

print(f"Input features shape: {input_features_dense.shape}")

In [ ]:

input_for_model = input_features_dense.reshape(input_features_dense.shape[0], 1, input_features_dense.shape[1])

print(f"Input for model shape: {input_for_model.shape}")

In [ ]:

prediction_probabilities = model.predict(input_for_model)


predicted_class_index = np.argmax(prediction_probabilities, axis=1)[0]

predicted_status = label_encoder.inverse_transform([predicted_class_index])[0]

print(f"Prediction Probabilities: {prediction_probabilities}")
print(f"Predicted Class Index: {predicted_class_index}")
print(f"Predicted Mental Health Status: {predicted_status}")